# Wine Explorer: demo

Every call goes through **Nginx on port 80**, exactly like a real client. Before running:

```bash
make docker-run   # neo4j + api + nginx
make seed         # WineEnthusiast + X-Wines into Neo4j
make gds          # similarity, flavour families, embeddings, classifier
```
`/api/ask` also needs `LLM_API_URL`, `LLM_API_KEY` and `LLM_MODEL` in `.env`.

In [1]:
import json
import os
import urllib.error
import urllib.parse
import urllib.request

BASE = "http://localhost"  # Nginx, the only public entry point


def call(path, body=None, **params):
    """GET (or POST when a body is given) through Nginx; returns (status, json)."""
    url = BASE + urllib.parse.quote(path) + ("?" + urllib.parse.urlencode(params) if params else "")
    headers = {"Content-Type": "application/json"}
    if os.getenv("API_KEY"):
        headers["X-API-Key"] = os.environ["API_KEY"]
    data = json.dumps(body).encode() if body is not None else None
    request = urllib.request.Request(url, data=data, headers=headers, method="POST" if body else "GET")
    try:
        with urllib.request.urlopen(request, timeout=180) as response:
            return response.status, json.load(response)
    except urllib.error.HTTPError as error:
        return error.code, json.load(error)


def show(rows, columns, limit=10):
    """Print rows as a small text table."""
    rows = rows[:limit]
    cell = lambda row, col: str(row.get(col, ""))[:45]
    widths = {c: max([len(c)] + [len(cell(r, c)) for r in rows]) for c in columns}
    print(" | ".join(c.ljust(widths[c]) for c in columns))
    print("-+-".join("-" * widths[c] for c in columns))
    for row in rows:
        print(" | ".join(cell(row, c).ljust(widths[c]) for c in columns))


call("/health")

(200, {'status': 'ok', 'neo4j': 'up'})

## 1. Filtered, sorted, paginated list
The query is built only from the filters that are given, and starts from the most selective node (here the grape). `has_more` comes from fetching one extra row instead of a second `COUNT` query.

In [2]:
status, page = call("/api/wines", variety="Riesling", country="Germany", min_points=92, max_price=40, limit=5)
show(page["items"], ["id", "title", "points", "price", "vintage"])
print("\nhas_more:", page["has_more"])

id    | title                                         | points | price | vintage
------+-----------------------------------------------+--------+-------+--------
78306 | Dr. Loosen 2014 Ürziger Würzgarten Spätlese R | 95     | 34.0  | 2014   
87146 | S.A. Prüm 2011 Wehlener Sonnenuhr Spätlese Gr | 95     | 35.0  | 2011   
15205 | Matthias Müller 2012 Alte Reben Bopparder Ham | 94     | 33.0  | 2012   
15788 | Domdechant Werner 2013 Hochheimer Domdechaney | 94     | 37.0  | 2013   
26431 | Schäfer-Fröhlich 2013 Vulkangestein Trocken R | 94     | 39.0  | 2013   

has_more: True


## 2. Best value
Points per dollar, among wines with at least `min_points`.

In [3]:
status, page = call("/api/wines/value", max_price=20, min_points=90, limit=5)
show(page["items"], ["title", "points", "price", "value_score"])

title                                         | points | price | value_score
----------------------------------------------+--------+-------+------------
Herdade dos Machados 2012 Toutalga Red (Alent | 91     | 7.0   | 13.0       
Snoqualmie 2006 Winemaker's Select Riesling ( | 91     | 8.0   | 11.375     
Aveleda 2013 Quinta da Aveleda Estate Bottled | 90     | 8.0   | 11.25      
Rothbury Estate 2001 Chardonnay (South Easter | 90     | 8.0   | 11.25      
Esser Cellars 2001 Chardonnay (California)    | 90     | 8.0   | 11.25      


## 3. One wine in context
Location chain (ragged: some wines have no region), critic and tasting notes, all in one query with pattern comprehensions.

In [4]:
status, page = call("/api/wines", variety="Pinot Noir", country="US", min_points=95, limit=1)
wine_id = page["items"][0]["id"]
status, wine = call(f"/api/wines/{wine_id}")
{k: wine[k] for k in ["title", "points", "price", "taster", "location", "notes"]}

{'title': 'Williams Selyem 2009 Precious Mountain Vineyard Pinot Noir (Sonoma Coast)',
 'points': 99,
 'price': 94.0,
 'taster': None,
 'location': ['Sonoma Coast', 'California', 'US'],
 'notes': ['raspberry', 'baking spice', 'smoke', 'cherry', 'mineral', 'oak']}

## 4. Similar wines (GDS Node Similarity)
Read from the precomputed `SIMILAR_TO` relationships (Jaccard on shared tasting notes, very common notes left out). With `max_price` it becomes "something like this, but cheaper".

In [5]:
status, similar = call(f"/api/wines/{wine_id}/similar", limit=5)
print("method:", similar["method"])
show(similar["items"], ["title", "price", "points", "score"])

status, cheaper = call(f"/api/wines/{wine_id}/similar", max_price=30, limit=5)
print("\nUnder $30:")
show(cheaper["items"], ["title", "price", "points", "score"])

method: gds_node_similarity
title                                         | price | points | score             
----------------------------------------------+-------+--------+-------------------
MacPhail 2009 Raye's Hill Pinot Noir (Anderso | 49.0  | 94     | 0.8333333333333334
Storm 2008 Le Bon Climat Vineyard Pinot Noir  | 40.0  | 91     | 0.8333333333333334
Failla 2010 Hirsch Vineyard Pinot Noir (Sonom | 70.0  | 97     | 0.7142857142857143
Sinor-LaVallee 2010 Talley-Rincon Vineyard Pi | 40.0  | 92     | 0.7142857142857143
Freestone 2008 Quarter Moon Vineyard Pinot No | 100.0 | 94     | 0.6666666666666666

Under $30:
title                                         | price | points | score             
----------------------------------------------+-------+--------+-------------------
De Loach 2007 Pinot Noir (Russian River Valle | 24.0  | 86     | 0.6666666666666666


## 5. Predicted quality tier (FastRP + logistic regression)
The model only sees the wine's graph embedding (notes, grape, place, producer), never its points or price. The actual points are returned next to the prediction so the user can judge it.

In [6]:
call(f"/api/wines/{wine_id}/predicted-tier")[1]

{'wine_id': 1557,
 'predicted_tier': 'high',
 'probability': 0.6641649305800614,
 'actual_points': 99}

## 6. Region profile
A region is identified by `(name, province)`. An ambiguous name answers 409 with the candidate provinces.

In [7]:
status, profile = call("/api/regions/Napa Valley/profile", province="California")
print(profile["region"], "|", profile["wine_count"], "wines | avg points", profile["avg_points"], "| avg price", profile["avg_price"])
show(profile["top_varieties"], ["name", "wines"], 5)
print()
show(profile["top_notes"], ["name", "wines"], 5)

Napa Valley | 4174 wines | avg points 89.05031145184466 | avg price 55.56714628297366
name                     | wines
-------------------------+------
Cabernet Sauvignon       | 1658 
Chardonnay               | 373  
Sauvignon Blanc          | 334  
Merlot                   | 333  
Bordeaux-style Red Blend | 310  

name         | wines
-------------+------
oak          | 1510 
blackberry   | 1363 
cherry       | 920  
fruit        | 905  
blackcurrant | 816  


## 7. Taster bias
How one critic scores each grape compared with all the other critics.

In [8]:
status, bias = call("/api/tasters/Roger Voss/bias", min_reviews=50, limit=8)
print(bias["taster"], "|", bias["total_reviews"], "reviews | overall deviation", bias["overall_deviation"])
show(bias["items"], ["variety", "reviews", "taster_avg", "others_avg", "deviation"])

Roger Voss | 20938 reviews | overall deviation 0.5719439625260977
variety         | reviews | taster_avg        | others_avg        | deviation          
----------------+---------+-------------------+-------------------+--------------------
Alvarinho       | 120     | 88.875            | 83.66666666666667 | 5.208333333333329  
Arinto          | 69      | 87.15942028985506 | 83.0              | 4.159420289855063  
Tannat          | 56      | 91.48214285714285 | 87.44329896907219 | 4.038843888070659  
Portuguese Red  | 2278    | 88.84372256365242 | 86.0              | 2.8437225636524204 
Champagne Blend | 1080    | 90.09629629629632 | 87.84615384615381 | 2.250142450142505  
Malbec-Merlot   | 74      | 88.5810810810811  | 86.5              | 2.081081081081095  
Malbec          | 410     | 89.66829268292685 | 87.65415821501021 | 2.0141344679166338 
Sparkling Blend | 185     | 86.39999999999999 | 88.16726748377789 | -1.7672674837778999


## 8. Food pairing (joins the two sources)
Food comes from X-Wines, wines from WineEnthusiast: they meet at the grape variety.

In [9]:
status, page = call("/api/pairings", food="beef", max_price=30, min_points=90, limit=8)
show(page["items"], ["title", "variety", "price", "points", "pairing_weight"])

title                                         | variety      | price | points | pairing_weight
----------------------------------------------+--------------+-------+--------+---------------
Adriano Marco & Vittorio 2013 Sanadaive  (Bar | Nebbiolo     | 30.0  | 94     | 1.0           
Villa Cafaggio 2009 Riserva  (Chianti Classic | Sangiovese   | 30.0  | 94     | 1.0           
Tamarack Cellars 2012 Dionysus Vineyard Petit | Petit Verdot | 30.0  | 94     | 1.0           
Domaine Berthoumieu 2007 Charles de Batz Tann | Tannat       | 25.0  | 94     | 1.0           
Adriano Marco & Vittorio 2010 Sanadaive  (Bar | Nebbiolo     | 28.0  | 94     | 1.0           
Castelli del Grevepesa 2010 Lamole Gran Selez | Sangiovese   | 28.0  | 94     | 1.0           
Château d'Aydie 2010 Tannat (Madiran)         | Tannat       | 26.0  | 94     | 1.0           
Naggiar 2014 Estate Petite Sirah (Sierra Foot | Petite Sirah | 29.0  | 94     | 1.0           


## 9. Flavour families (GDS Louvain)
Communities of tasting notes that appear together more often than chance.

In [10]:
status, styles = call("/api/styles")
for style in styles["items"]:
    print(f"style {style['id']} ({style['size']} notes):", ", ".join(style["top_notes"][:8]))

status, style = call("/api/styles/2")
print("\nExample wines for style 2:")
show(style["example_wines"], ["title", "variety", "points"], 5)

style 0 (48 notes): apple, toast, citrus, mineral, peach, floral, lemon, pear
style 2 (41 notes): spice, oak, blackberry, plum, vanilla, black cherry, blackcurrant, wood
style 1 (31 notes): cherry, herb, raspberry, earth, strawberry, clove, red fruit, cinnamon



Example wines for style 2:
title                                         | variety            | points
----------------------------------------------+--------------------+-------
Casa di Terra 2011 Maronea  (Bolgheri)        | Red Blend          | 87    
Viñedos de Paganos 2009 La Nieta  (Rioja)     | Tempranillo        | 94    
Campo Eliseo 2009  Toro                       | Tinta de Toro      | 95    
Falcone 2013 Mia's Vineyard Cabernet Sauvigno | Cabernet Sauvignon | 95    
Kanonkop 2013 Estate Wine Black Label Pinotag | Pinotage           | 93    


## 10. Questions in plain English (NL -> Cypher)
The response keeps the generated Cypher, the rows (verified, from the database) and the summary (AI-generated from those rows only) apart.

In [11]:
status, answer = call("/api/ask", body={"question": "Which 5 wineries have the most wines rated 95 points or more?"})
print(status, answer["status"])
print(answer["cypher"], "\n")
show(answer["rows"], list(answer["rows"][0].keys()) if answer["rows"] else [])
print("\nSummary:", answer["summary"])
print("Provenance:", answer["provenance"])

200 ok
MATCH (w:Wine)-[:MADE_BY]->(winery:Winery) WHERE w.points >= 95
RETURN winery.name AS winery, count(w) AS wine_count ORDER BY wine_count DESC LIMIT 5 

winery                 | wine_count
-----------------------+-----------
Williams Selyem        | 54        
Cayuse                 | 27        
Louis Jadot            | 25        
Domaine Zind-Humbrecht | 20        
Lynmar                 | 20        

Summary: The five wineries with the most wines rated 95 points or higher are Williams Selyem, Cayuse, Louis Jadot, Domaine Zind‑Humbrecht, and Lynmar.
Provenance: {'rows': 'verified (database)', 'summary': 'AI-generated from rows only'}


In [12]:
for question in ["Delete all wines from France", "What will the weather be in Paris tomorrow?"]:
    status, answer = call("/api/ask", body={"question": question})
    print(status, answer["status"], "-", question, "->", answer["reason"])

400 rejected - Delete all wines from France -> the question asks to change data


400 rejected - What will the weather be in Paris tomorrow? -> the question is not about the wine graph


## 11. Errors are clean JSON
No stack traces reach the client.

In [13]:
for path, params in [("/api/wines/999999999", {}), ("/api/wines", {"min_points": 200})]:
    status, body = call(path, **params)
    print(status, json.dumps(body)[:160])

404 {"error": "Wine '999999999' not found"}
422 {"detail": [{"type": "less_than_equal", "loc": ["query", "min_points"], "msg": "Input should be less than or equal to 100", "input": "200", "ctx": {"le": 100}}]
